# ETL Silver → Gold: Migração para Modelo Dimensional

Este notebook realiza a extração, transformação e carga (ETL) dos dados da camada **Silver** (tabela `lancamentos` no schema `public`) para a camada **Gold** (modelo dimensional Star Schema no schema `gold`).

## 1. Importações e Configurações

In [22]:
import psycopg2
from psycopg2 import sql
import pandas as pd
from datetime import datetime

# Configurações de conexão com o banco de dados
DB_CONFIG = {
    'host': 'localhost',
    'port': 5433,
    'database': 'filmes_db',
    'user': 'user',
    'password': 'password'
}

print("✓ Bibliotecas importadas com sucesso!")
print(f"✓ Configuração: {DB_CONFIG['database']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}")

✓ Bibliotecas importadas com sucesso!
✓ Configuração: filmes_db@localhost:5433


## 2. Criação do Schema Gold e Todas as Tabelas (usando ddl.sql)

In [23]:
# Carregar e executar o script DDL do arquivo
import os

# Caminho para o arquivo DDL
ddl_file_path = '../Data-Layer/gold/sql/ddl.sql'

try:
    # Ler o arquivo DDL
    with open(ddl_file_path, 'r', encoding='utf-8') as file:
        ddl_script = file.read()
    
    print(f"✓ Arquivo DDL carregado: {ddl_file_path}")
    print(f"✓ Tamanho do script: {len(ddl_script)} caracteres\n")
    
    # Executar o DDL
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    cursor.execute(ddl_script)
    conn.commit()
    
    print("=" * 60)
    print("✓ Schema 'gold' criado com sucesso!")
    print("✓ Tabelas dimensionais criadas:")
    print("  - gold.DIM_DISTRIBUIDORA")
    print("  - gold.DIM_FILME")
    print("  - gold.DIM_DATA_LANCAMENTO")
    print("✓ Tabela fato criada:")
    print("  - gold.FAT_LANCAMENTO")
    print("✓ Índices criados para otimização")
    print("✓ Chaves estrangeiras configuradas:")
    print("  - srk_filme_fk → DIM_FILME")
    print("  - srk_dlan_fk → DIM_DATA_LANCAMENTO")
    print("  - srk_dist_fk → DIM_DISTRIBUIDORA")
    print("=" * 60)
    
    cursor.close()
    conn.close()
    
except FileNotFoundError:
    print(f"✗ Erro: Arquivo DDL não encontrado em '{ddl_file_path}'")
    print("  Certifique-se de que o caminho está correto.")
except Exception as e:
    print(f"✗ Erro ao executar DDL: {e}")
    if conn:
        conn.rollback()
        conn.close()

✓ Arquivo DDL carregado: ../Data-Layer/gold/sql/ddl.sql
✓ Tamanho do script: 1895 caracteres

✓ Schema 'gold' criado com sucesso!
✓ Tabelas dimensionais criadas:
  - gold.DIM_DISTRIBUIDORA
  - gold.DIM_FILME
  - gold.DIM_DATA_LANCAMENTO
✓ Tabela fato criada:
  - gold.FAT_LANCAMENTO
✓ Índices criados para otimização
✓ Chaves estrangeiras configuradas:
  - srk_filme_fk → DIM_FILME
  - srk_dlan_fk → DIM_DATA_LANCAMENTO
  - srk_dist_fk → DIM_DISTRIBUIDORA


## 3. Extração de Dados da Camada Silver

In [24]:
# Extrair dados da tabela lancamentos no schema public
try:
    # Usando connection string SQLAlchemy para evitar warnings
    DATABASE_URL = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    
    query = """
    SELECT 
        data_lancamento,
        titulo_original,
        cpb_roe,
        tipo_obra,
        pais_obra,
        publico_total,
        renda_total,
        distribuidora,
        registro_distribuidora,
        cnpj_distribuidora,
        ano_lancamento,
        mes_lancamento,
        dia_lancamento
    FROM public.lancamentos
    WHERE data_lancamento IS NOT NULL
    """
    
    df_silver = pd.read_sql_query(query, DATABASE_URL)
    
    print(f"✓ Dados extraídos da camada Silver: {len(df_silver)} registros")
    print(f"✓ Colunas: {list(df_silver.columns)}")
    print(f"\n📊 Preview dos dados:")
    display(df_silver.head())
    
except Exception as e:
    print(f"✗ Erro ao extrair dados: {e}")

✗ Erro ao extrair dados: Using URI string without sqlalchemy installed.


## 4. Transformação e Carga - DIM_DISTRIBUIDORA

In [25]:
# Extrair distribuidoras únicas
dim_distribuidora = df_silver[['registro_distribuidora', 'distribuidora', 'cnpj_distribuidora']].drop_duplicates()

print(f"Total de distribuidoras únicas: {len(dim_distribuidora)}")

# Inserir na tabela DIM_DISTRIBUIDORA
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO gold.DIM_DISTRIBUIDORA (registro_distribuidora, distribuidora, cnpj_distribuidora)
    VALUES (%s, %s, %s)
    RETURNING srk_dist_pk
    """
    
    registros_inseridos = 0
    for _, row in dim_distribuidora.iterrows():
        cursor.execute(insert_query, (
            row['registro_distribuidora'],
            row['distribuidora'],
            row['cnpj_distribuidora']
        ))
        registros_inseridos += 1
    
    conn.commit()
    cursor.close()
    conn.close()
    
    print(f"✓ {registros_inseridos} distribuidoras inseridas em gold.DIM_DISTRIBUIDORA")
    
except Exception as e:
    print(f"✗ Erro ao inserir distribuidoras: {e}")
    if conn:
        conn.rollback()
        conn.close()

Total de distribuidoras únicas: 378
✓ 378 distribuidoras inseridas em gold.DIM_DISTRIBUIDORA
✓ 378 distribuidoras inseridas em gold.DIM_DISTRIBUIDORA


## 5. Transformação e Carga - DIM_FILME

In [26]:
# Extrair filmes únicos
dim_filme = df_silver[['titulo_original', 'tipo_obra', 'pais_obra', 'cpb_roe']].drop_duplicates()

print(f"Total de filmes únicos: {len(dim_filme)}")

# Verificar se há valores nulos que podem causar problemas
print(f"Valores nulos em titulo_original: {dim_filme['titulo_original'].isnull().sum()}")
print(f"Valores nulos em cpb_roe: {dim_filme['cpb_roe'].isnull().sum()}")

# Inserir na tabela DIM_FILME
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO gold.DIM_FILME (titulo_original, tipo_obra, pais_obra, cpb_roe)
    VALUES (%s, %s, %s, %s)
    RETURNING srk_filme_pk
    """
    
    registros_inseridos = 0
    erros = 0
    
    for idx, row in dim_filme.iterrows():
        try:
            cursor.execute(insert_query, (
                row['titulo_original'],
                row['tipo_obra'],
                row['pais_obra'],
                row['cpb_roe']
            ))
            registros_inseridos += 1
        except Exception as e:
            erros += 1
            if erros <= 3:  # Mostrar apenas os 3 primeiros erros
                print(f"  ⚠ Erro ao inserir: {e}")
                print(f"     Dados: {row['titulo_original'][:50]}...")
    
    conn.commit()
    cursor.close()
    conn.close()
    
    print(f"✓ {registros_inseridos} filmes inseridos em gold.DIM_FILME")
    if erros > 0:
        print(f"⚠ {erros} erros encontrados durante a inserção")
    
except Exception as e:
    print(f"✗ Erro ao inserir filmes: {e}")
    import traceback
    traceback.print_exc()
    if 'conn' in locals():
        conn.rollback()
        conn.close()

Total de filmes únicos: 6495
Valores nulos em titulo_original: 0
Valores nulos em cpb_roe: 0
✓ 6495 filmes inseridos em gold.DIM_FILME
✓ 6495 filmes inseridos em gold.DIM_FILME


### 5.1. Diagnóstico: Verificando dados únicos

In [27]:
# Diagnóstico: verificar estrutura e valores dos dados
print("=" * 60)
print("🔍 DIAGNÓSTICO DE DADOS")
print("=" * 60)

print("\n1. Verificando DataFrame carregado:")
print(f"   - Shape: {df_silver.shape}")
print(f"   - Colunas: {list(df_silver.columns)}")

print("\n2. Verificando valores de 'cpb_roe':")
print(f"   - Valores únicos: {df_silver['cpb_roe'].nunique()}")
print(f"   - Valores nulos: {df_silver['cpb_roe'].isnull().sum()}")
print(f"   - Primeiros 5 valores:")
display(df_silver[['titulo_original', 'cpb_roe']].head(10))

print("\n3. Verificando duplicatas em filmes:")
print(f"   - Total de registros: {len(df_silver)}")
print(f"   - Filmes únicos (titulo + cpb_roe): {len(df_silver[['titulo_original', 'cpb_roe']].drop_duplicates())}")

print("\n4. Verificando valores de distribuidora:")
print(f"   - CNPJ únicos: {df_silver['cnpj_distribuidora'].nunique()}")
print(f"   - CNPJ nulos: {df_silver['cnpj_distribuidora'].isnull().sum()}")

print("\n5. Testando extração de dimensões:")
test_dim_filme = df_silver[['titulo_original', 'tipo_obra', 'pais_obra', 'cpb_roe']].drop_duplicates()
print(f"   - DIM_FILME únicos: {len(test_dim_filme)}")
display(test_dim_filme.head())

🔍 DIAGNÓSTICO DE DADOS

1. Verificando DataFrame carregado:
   - Shape: (33265, 13)
   - Colunas: ['data_lancamento', 'titulo_original', 'cpb_roe', 'tipo_obra', 'pais_obra', 'publico_total', 'renda_total', 'distribuidora', 'registro_distribuidora', 'cnpj_distribuidora', 'ano_lancamento', 'mes_lancamento', 'dia_lancamento']

2. Verificando valores de 'cpb_roe':
   - Valores únicos: 6482
   - Valores nulos: 0
   - Primeiros 5 valores:


,titulo_original,cpb_roe
0,aldo baldin - uma vida pela música,B2400467800000
1,death of a unicorn,E2500223800000
2,guns up,E2500153200000
3,materialists,E2500150300000
4,nada,B2300179900000
5,the ritual,E2500153400000
6,bts army: forever we are young,E2500271700000
7,o deserto de akin,B2500117300000
8,in the blind spot,E2500205000000
9,monsieur aznavour,E2500146000000



3. Verificando duplicatas em filmes:
   - Total de registros: 33265
   - Filmes únicos (titulo + cpb_roe): 6495

4. Verificando valores de distribuidora:
   - CNPJ únicos: 369
   - CNPJ nulos: 0

5. Testando extração de dimensões:
   - DIM_FILME únicos: 6495


,titulo_original,tipo_obra,pais_obra,cpb_roe
0,aldo baldin - uma vida pela música,documentário,brasil,B2400467800000
1,death of a unicorn,ficção,estados unidos,E2500223800000
2,guns up,ficção,estados unidos,E2500153200000
3,materialists,ficção,estados unidos,E2500150300000
4,nada,ficção,brasil,B2300179900000


## 6. Transformação e Carga - DIM_DATA_LANCAMENTO

In [28]:
# Extrair datas únicas
dim_data = df_silver[['dia_lancamento', 'mes_lancamento', 'ano_lancamento']].drop_duplicates()

print(f"Total de datas únicas: {len(dim_data)}")

# Inserir na tabela DIM_DATA_LANCAMENTO
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO gold.DIM_DATA_LANCAMENTO (dia, mes, ano)
    VALUES (%s, %s, %s)
    RETURNING srk_dlan_pk
    """
    
    registros_inseridos = 0
    for _, row in dim_data.iterrows():
        cursor.execute(insert_query, (
            int(row['dia_lancamento']),
            int(row['mes_lancamento']),
            int(row['ano_lancamento'])
        ))
        registros_inseridos += 1
    
    conn.commit()
    cursor.close()
    conn.close()
    
    print(f"✓ {registros_inseridos} datas inseridas em gold.DIM_DATA_LANCAMENTO")
    
except Exception as e:
    print(f"✗ Erro ao inserir datas: {e}")
    if conn:
        conn.rollback()
        conn.close()

Total de datas únicas: 1156
✓ 1156 datas inseridas em gold.DIM_DATA_LANCAMENTO
✓ 1156 datas inseridas em gold.DIM_DATA_LANCAMENTO


## 7. Carga da Tabela Fato - FAT_LANCAMENTO

In [30]:
# Popular tabela fato relacionando com as dimensões
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    # Primeiro, verificar quantos registros seriam inseridos
    check_query = """
    SELECT COUNT(*)
    FROM public.lancamentos l
    INNER JOIN gold.DIM_FILME f 
        ON l.titulo_original = f.titulo_original 
        AND l.cpb_roe = f.cpb_roe
    INNER JOIN gold.DIM_DATA_LANCAMENTO d 
        ON l.dia_lancamento = d.dia 
        AND l.mes_lancamento = d.mes 
        AND l.ano_lancamento = d.ano
    INNER JOIN gold.DIM_DISTRIBUIDORA dist 
        ON l.cnpj_distribuidora = dist.cnpj_distribuidora
    WHERE l.data_lancamento IS NOT NULL
    """
    
    cursor.execute(check_query)
    count_expected = cursor.fetchone()[0]
    print(f"📊 Registros esperados para inserção: {count_expected}")
    
    if count_expected == 0:
        print("\n⚠ ATENÇÃO: Nenhum registro será inserido!")
        print("   Verificando possíveis problemas de JOIN:\n")
        
        # Verificar cada JOIN separadamente
        cursor.execute("""
            SELECT 
                COUNT(*) as total_silver,
                COUNT(DISTINCT l.titulo_original) as titulos_unicos,
                COUNT(DISTINCT l.cpb_roe) as cpb_unicos
            FROM public.lancamentos l
            WHERE l.data_lancamento IS NOT NULL
        """)
        silver_stats = cursor.fetchone()
        print(f"   Silver: {silver_stats[0]} registros, {silver_stats[1]} títulos, {silver_stats[2]} CPB/ROE únicos")
        
        cursor.execute("SELECT COUNT(*) FROM gold.DIM_FILME")
        dim_filme_count = cursor.fetchone()[0]
        print(f"   DIM_FILME: {dim_filme_count} registros")
        
        cursor.execute("SELECT COUNT(*) FROM gold.DIM_DATA_LANCAMENTO")
        dim_data_count = cursor.fetchone()[0]
        print(f"   DIM_DATA_LANCAMENTO: {dim_data_count} registros")
        
        cursor.execute("SELECT COUNT(*) FROM gold.DIM_DISTRIBUIDORA")
        dim_dist_count = cursor.fetchone()[0]
        print(f"   DIM_DISTRIBUIDORA: {dim_dist_count} registros")
        
        # Verificar JOIN com DIM_FILME
        cursor.execute("""
            SELECT COUNT(*)
            FROM public.lancamentos l
            INNER JOIN gold.DIM_FILME f 
                ON l.titulo_original = f.titulo_original 
                AND l.cpb_roe = f.cpb_roe
            WHERE l.data_lancamento IS NOT NULL
        """)
        join_filme = cursor.fetchone()[0]
        print(f"\n   JOIN com DIM_FILME: {join_filme} matches")
        
        # Verificar JOIN com DIM_DATA
        cursor.execute("""
            SELECT COUNT(*)
            FROM public.lancamentos l
            INNER JOIN gold.DIM_DATA_LANCAMENTO d 
                ON l.dia_lancamento = d.dia 
                AND l.mes_lancamento = d.mes 
                AND l.ano_lancamento = d.ano
            WHERE l.data_lancamento IS NOT NULL
        """)
        join_data = cursor.fetchone()[0]
        print(f"   JOIN com DIM_DATA_LANCAMENTO: {join_data} matches")
        
        # Verificar JOIN com DIM_DISTRIBUIDORA
        cursor.execute("""
            SELECT COUNT(*)
            FROM public.lancamentos l
            INNER JOIN gold.DIM_DISTRIBUIDORA dist 
                ON l.cnpj_distribuidora = dist.cnpj_distribuidora
            WHERE l.data_lancamento IS NOT NULL
        """)
        join_dist = cursor.fetchone()[0]
        print(f"   JOIN com DIM_DISTRIBUIDORA: {join_dist} matches")
    
    # Query para inserir fatos com lookup nas dimensões
    insert_query = """
    INSERT INTO gold.FAT_LANCAMENTO (srk_filme_fk, srk_dlan_fk, srk_dist_fk, publico_total, renda_total)
    SELECT 
        f.srk_filme_pk,
        d.srk_dlan_pk,
        dist.srk_dist_pk,
        l.publico_total,
        l.renda_total
    FROM public.lancamentos l
    INNER JOIN gold.DIM_FILME f 
        ON l.titulo_original = f.titulo_original 
        AND l.cpb_roe = f.cpb_roe
    INNER JOIN gold.DIM_DATA_LANCAMENTO d 
        ON l.dia_lancamento = d.dia 
        AND l.mes_lancamento = d.mes 
        AND l.ano_lancamento = d.ano
    INNER JOIN gold.DIM_DISTRIBUIDORA dist 
        ON l.cnpj_distribuidora = dist.cnpj_distribuidora
    WHERE l.data_lancamento IS NOT NULL
    """
    
    cursor.execute(insert_query)
    registros_inseridos = cursor.rowcount
    conn.commit()
    
    print(f"\n✓ {registros_inseridos} registros inseridos em gold.FAT_LANCAMENTO")
    print("✓ Relacionamentos estabelecidos com todas as dimensões")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Erro ao popular tabela fato: {e}")
    import traceback
    traceback.print_exc()
    if 'conn' in locals():
        conn.rollback()
        conn.close()

📊 Registros esperados para inserção: 40458

✓ 40458 registros inseridos em gold.FAT_LANCAMENTO
✓ Relacionamentos estabelecidos com todas as dimensões

✓ 40458 registros inseridos em gold.FAT_LANCAMENTO
✓ Relacionamentos estabelecidos com todas as dimensões


## 8. Validação e Estatísticas

In [ ]:
# Verificar contagens e integridade dos dados
try:
    conn = psycopg2.connect(**DB_CONFIG)
    
    queries = {
        'DIM_DISTRIBUIDORA': 'SELECT COUNT(*) FROM gold.DIM_DISTRIBUIDORA',
        'DIM_FILME': 'SELECT COUNT(*) FROM gold.DIM_FILME',
        'DIM_DATA_LANCAMENTO': 'SELECT COUNT(*) FROM gold.DIM_DATA_LANCAMENTO',
        'FAT_LANCAMENTO': 'SELECT COUNT(*) FROM gold.FAT_LANCAMENTO',
        'Silver (origem)': 'SELECT COUNT(*) FROM public.lancamentos WHERE data_lancamento IS NOT NULL'
    }
    
    print("=" * 60)
    print("📊 ESTATÍSTICAS DO ETL SILVER → GOLD")
    print("=" * 60)
    
    for tabela, query in queries.items():
        df_count = pd.read_sql_query(query, conn)
        count = df_count.iloc[0, 0]
        print(f"{tabela:.<40} {count:>10,} registros")
    
    print("=" * 60)
    
    # Exemplo de consulta analítica no modelo dimensional
    query_exemplo = """
    SELECT 
        f.titulo_original,
        f.tipo_obra,
        d.distribuidora,
        dt.ano,
        ft.publico_total,
        ft.renda_total
    FROM gold.FAT_LANCAMENTO ft
    INNER JOIN gold.DIM_FILME f ON ft.srk_filme_fk = f.srk_filme_pk
    INNER JOIN gold.DIM_DISTRIBUIDORA d ON ft.srk_dist_fk = d.srk_dist_pk
    INNER JOIN gold.DIM_DATA_LANCAMENTO dt ON ft.srk_dlan_fk = dt.srk_dlan_pk
    ORDER BY ft.renda_total DESC
    LIMIT 10
    """
    
    df_exemplo = pd.read_sql_query(query_exemplo, conn)
    
    print("\n🎬 TOP 10 LANÇAMENTOS POR RENDA (Modelo Dimensional):")
    display(df_exemplo)
    
    conn.close()
        
except Exception as e:
    print(f"✗ Erro ao validar dados: {e}")
    if conn:
        conn.close()

📊 ESTATÍSTICAS DO ETL SILVER → GOLD
DIM_DISTRIBUIDORA.......................        378 registros
DIM_FILME...............................      6,495 registros
DIM_DATA_LANCAMENTO.....................      1,156 registros
FAT_LANCAMENTO..........................     40,458 registros
Silver (origem).........................     39,918 registros

🎬 TOP 10 LANÇAMENTOS POR RENDA (Modelo Dimensional):


C:\Users\alvea\AppData\Local\Temp\ipykernel_24424\3080591335.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql_query(query, conn)
C:\Users\alvea\AppData\Local\Temp\ipykernel_24424\3080591335.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_exemplo = pd.read_sql_query(query_exemplo, conn)


,titulo_original,tipo_obra,distribuidora,ano,publico_total,renda_total
0,inside out 2,animação,the walt disney company (brasil) ltda.,2024,22471805,4.444441e+08
1,inside out 2,animação,the walt disney company (brasil) ltda.,2024,22471805,4.444441e+08
2,inside out 2,animação,the walt disney company (brasil) ltda.,2024,22471805,4.444441e+08
3,inside out 2,animação,the walt disney company (brasil) ltda.,2024,22471805,4.444441e+08
4,inside out 2,animação,the walt disney company (brasil) ltda.,2024,22471805,4.444441e+08
5,inside out 2,animação,the walt disney company (brasil) ltda.,2024,22471805,4.444441e+08
6,avengers: endgame,ficção,the walt disney company (brasil) ltda.,2019,19656475,3.386250e+08
7,avengers: endgame,ficção,the walt disney company (brasil) ltda.,2019,19656475,3.386250e+08
8,avengers: endgame,ficção,the walt disney company (brasil) ltda.,2019,19656475,3.386250e+08
9,avengers: endgame,ficção,the walt disney company (brasil) ltda.,2019,19656475,3.386250e+08



✅ ETL SILVER → GOLD CONCLUÍDO COM SUCESSO!
